# FM Checkpoint Sampling (CIFAR10)

Dieses Notebook laedt ein trainiertes Flow-Matching-Checkpoint und sampelt 5 Bilder.

In [ ]:
from pathlib import Path
import sys

import torch
from IPython.display import Image as DisplayImage, display


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root (pyproject.toml not found).")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from flowguard.flow_matching import load_flow_matching_checkpoint, sample_images  # noqa: E402

PROJECT_ROOT

In [ ]:
checkpoint_path = PROJECT_ROOT / "runs" / "flow_matching" / "cifar10_notebook" / "checkpoint_latest.pt"
output_path = PROJECT_ROOT / "runs" / "flow_matching" / "cifar10_notebook" / "samples" / "sample_5_from_notebook.png"

device = "cuda" if torch.cuda.is_available() else "cpu"
sample_count = 5
ode_method = "midpoint"
step_size = 0.01
seed = 42

if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

checkpoint_path, output_path, device

In [ ]:
checkpoint = load_flow_matching_checkpoint(checkpoint_path, device=device)

saved_path = sample_images(
    checkpoint.model,
    checkpoint.dataset_config,
    output_path=output_path,
    device=device,
    sample_count=sample_count,
    ode_method=ode_method,
    step_size=step_size,
    seed=seed,
)

print(f"Saved sample grid to: {saved_path}")
print(f"Dataset type: {checkpoint.dataset_config.dataset_type}")
saved_path

In [ ]:
if checkpoint.dataset_config.dataset_type == "tabular":
    print("Tabular dataset detected. Output was saved as a tensor file.")
else:
    display(DisplayImage(filename=str(output_path)))